### imports and pandas settings

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

In [2]:
pd.set_option("display.max_rows", None)

### set the path to data file

In [3]:
load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [4]:
batch_data = CF_OUTPUTS / "base_vs_thresholds" / "base_xgb_highthres_2026-05-06" / "annotated.csv"

### load data 
convert to string to fillna with whitespace
we drop:  
- target_risk (half of "risk_before") , and 
- "hltprhc"

we rename "original" in df_id to "xin" (HL standards)

In [5]:
# Load CSV
batch_df = pd.read_csv(batch_data)

In [6]:
# Feature columns
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [7]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

### All CFs

In [8]:
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,0.95,,,,,
1,0,cf_1,,,,,,,16.7643,,,0.9687,1,True,0.026,0.004
2,0,cf_2,,,,7,,,18.9745,,,0.8755,2,False,0.026,0.0707
3,0,cf_3,,,,6,,,18.4449,,,0.8979,2,False,0.026,0.0737
4,0,cf_4,,,6,,,,17.913,,,0.9203,2,True,0.026,0.0045
5,0,cf_5,,,,7,,,17.6252,,,0.9324,2,False,0.026,0.0424
6,0,cf_6,3,,,,,,16.4122,,,0.9836,2,True,0.026,0.0095
7,0,cf_7,,,,,,,16.0235,5,,1.0,2,True,0.026,0.0007
8,0,cf_8,,,6,7,,,18.9745,,,0.8755,3,False,0.026,0.1428
9,0,cf_9,,,,7,,,18.9745,7,,0.8755,3,True,0.026,0.0038


### Filtering Valid CFs

In [9]:
from cf_bench.utils import filter_valid_cfs

batch_df = filter_valid_cfs(batch_df)
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,0.95,,,,,
9,0,cf_1,,,,,,,16.7643,,,0.9687,1,True,0.026,0.004
10,0,cf_4,,,6,,,,17.913,,,0.9203,2,True,0.026,0.0045
11,0,cf_6,3,,,,,,16.4122,,,0.9836,2,True,0.026,0.0095
12,0,cf_7,,,,,,,16.0235,5,,1.0,2,True,0.026,0.0007
13,0,cf_9,,,,7,,,18.9745,7,,0.8755,3,True,0.026,0.0038
14,0,cf_10,3,,,,,,18.9619,3,,0.876,3,True,0.026,0.0117
1,1,xin,3,4,1,2,3,0,22.3757,0,1.06,,,,,
15,1,cf_3,,,,3,1,,22.3757,,,0.8796,3,True,0.2497,0.0534
16,1,cf_4,,,2,,1,,22.3757,,,0.8796,3,True,0.2497,0.0851


### select one CF

In [10]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation (prefers valid CFs without violations)
single_cf_df = select_one_cf_per_query(batch_df, prefer_no_violations=True)
single_cf_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,0.95,,,,,
9,0,cf_1,,,,,,,16.7643,,,0.9687,1,True,0.026,0.004
1,1,xin,3,4,1,2,3,0,22.3757,0,1.06,,,,,
10,1,cf_3,,,,3,1,,22.3757,,,0.8796,3,True,0.2497,0.0534
2,2,xin,5,3,1,1,4,0,22.694,7,1.25,,,,,
11,2,cf_3,,,,4,2,,22.6757,,,0.8754,3,True,0.218,0.0241
3,3,xin,3,3,6,6,2,0,24.3418,1,1.02,,,,,
4,4,xin,5,4,2,7,1,0,21.2585,3,1.15,,,,,
12,4,cf_8,,1,,,,,21.2585,,,0.9868,2,True,0.0811,0.0133
5,5,xin,2,2,1,2,4,0,27.9904,0,1.11,,,,,


In [11]:
from cf_bench.dice_adapters import DiceRecommender

recommender = DiceRecommender()

# Format a single query
for idx in range(0, 10):
    print(recommender.format_query(batch_df, query_index=idx))


=== Query index '0' ===
Task / Target: hltprhc
Query instance index: 0

Original instance:
  etfruit: 4
  eatveg: 3
  cgtsmok: 3
  alcfreq: 4
  slprl: 2
  paccnois: 0
  bmi: 18.9866
  dosprt: 0


=== Counterfactuals ===

--- cf_1 ---
Predicted risk: 0.004
Valid: True
Changes:
  bmi: 18.9866 → 16.7643

--- cf_4 ---
Predicted risk: 0.0045
Valid: True
Changes:
  cgtsmok: 3 → 6
  bmi: 18.9866 → 17.913

--- cf_6 ---
Predicted risk: 0.0095
Valid: True
Changes:
  etfruit: 4 → 3
  bmi: 18.9866 → 16.4122

--- cf_7 ---
Predicted risk: 0.0007
Valid: True
Changes:
  bmi: 18.9866 → 16.0235
  dosprt: 0 → 5

--- cf_9 ---
Predicted risk: 0.0038
Valid: True
Changes:
  alcfreq: 4 → 7
  bmi: 18.9866 → 18.9745
  dosprt: 0 → 7

--- cf_10 ---
Predicted risk: 0.0117
Valid: True
Changes:
  etfruit: 4 → 3
  bmi: 18.9866 → 18.9619
  dosprt: 0 → 3


=== Query index '1' ===
Task / Target: hltprhc
Query instance index: 1

Original instance:
  etfruit: 3
  eatveg: 4
  cgtsmok: 1
  alcfreq: 2
  slprl: 3
  paccnois: